# ML Evaluation: Keyword-to-Mood Classification

This notebook evaluates the keyword-to-mood classifier that powers the movie recommender's mood filtering system. Given a TMDB keyword like "murder" or "christmas", the classifier predicts which of 7 Ekman emotions the keyword evokes.

**Problem:** 68,000+ TMDB keywords need mood labels. Only 1,049 have manual single-label annotations. The classifier learns from these 1,049 and infers mood probabilities for the remaining ~67K.

**Approach:** Encode keywords as semantic embeddings (EmbeddingGemma-300M), then train a multi-class classifier on the 7 mood categories.

**Course requirements covered:**
1. Train/val/test split with stratification
2. Feature scaling (RobustScaler, fit on train only)
3. 5+ classifier comparison (scaled vs unscaled) with DummyClassifier baselines
4. Best model selection + confusion matrix + classification report
5. 10-fold cross-validation
6. KNN hyperparameter tuning (k=1..20)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, RobustScaler

DATA_DIR = Path("../../data")
TSV_PATH = DATA_DIR / "input" / "labeled_keywords.tsv"
OUTPUT_DIR = DATA_DIR / "output"

## 1. Data Loading

The labeled dataset contains the top 5,000 TMDB keywords by movie count, each manually assigned to one or more of the 7 Ekman moods. We use only the **single-label subset** (1,049 keywords) for clean multi-class classification.

In [ ]:
labeled = pd.read_csv(TSV_PATH, sep="\t")
single = labeled[labeled["assignment_type"] == "single"]
keywords = single["keyword_name"].tolist()
labels = single["assigned_moods"].tolist()

print(f"Total labeled keywords: {len(labeled)}")
print(f"  single-label: {(labeled['assignment_type'] == 'single').sum()}")
print(f"  multi-label:  {(labeled['assignment_type'] == 'multi').sum()}")
print(f"  unlabeled:    {(labeled['assignment_type'] == 'none').sum()}")
print(f"\nUsing {len(single)} single-label keywords for classification.")

## 2. Class Distribution

The dataset is **heavily imbalanced**. "Interested" has 6x more samples than "Disgusted". This affects classifier behavior: models may favor the majority class. We use `class_weight="balanced"` where supported and macro-averaged metrics to account for this.

In [ ]:
class_counts = single["assigned_moods"].value_counts()

fig, ax = plt.subplots(figsize=(10, 4))
colors = ["#21c354", "#1c83e1", "#ffa421", "#ff4b4b", "#803df5", "#a3a8b8", "#ff6b6b"]
class_counts.plot.barh(ax=ax, color=colors[:len(class_counts)])
ax.set_xlabel("Keywords")
ax.set_title("Class Distribution (single-label keywords)")
for i, (mood, count) in enumerate(class_counts.items()):
    ax.text(count + 3, i, str(count), va="center", fontsize=10)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(f"\nImbalance ratio (max/min): {class_counts.max() / class_counts.min():.1f}x")
print(f"Baseline accuracy (most_frequent): {class_counts.max() / class_counts.sum():.1%}")

## 3. Embeddings

We encode each keyword string into a dense vector using a pre-trained sentence transformer. This captures semantic meaning: "murder" and "serial killer" will be close in embedding space, even though they share no characters.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("google/embeddinggemma-300m")
embeddings = model.encode(keywords, show_progress_bar=True, normalize_embeddings=True)
print(f"Embedding shape: {embeddings.shape}")
print(f"Vector norm (should be ~1.0): {np.linalg.norm(embeddings[0]):.4f}")

## 4. Embedding Space Visualization (t-SNE)

Projecting the 256-dim embeddings into 2D to see whether mood classes form natural clusters. Clear separation suggests the embeddings carry mood-relevant information even before training a classifier.

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
emb_2d = tsne.fit_transform(embeddings)

mood_colors = {
    "Happy": "#21c354", "Interested": "#1c83e1", "Surprised": "#ffa421",
    "Sad": "#6c757d", "Disgusted": "#803df5", "Afraid": "#ff4b4b",
    "Angry": "#ff6b6b",
}

fig, ax = plt.subplots(figsize=(12, 8))
for mood, color in mood_colors.items():
    mask = np.array(labels) == mood
    ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1], c=color, label=mood,
               alpha=0.6, s=20, edgecolors="none")
ax.legend(title="Mood", fontsize=9)
ax.set_title("t-SNE Projection of Keyword Embeddings (colored by mood)")
ax.set_xticks([])
ax.set_yticks([])
plt.tight_layout()
plt.show()

## 5. Train / Validation / Test Split

- **80% train** (839) — for fitting classifiers
- **10% validation** (105) — for model selection and hyperparameter tuning
- **10% test** (105) — held out until final evaluation, used exactly once

All splits are stratified to preserve the class distribution.

In [ ]:
le = LabelEncoder()
y = le.fit_transform(labels)
label_names = list(le.classes_)

# 90% train+val, 10% test
x_tv, x_test, y_tv, y_test = train_test_split(
    embeddings, y, test_size=0.10, stratify=y, random_state=13,
)
# 80% train, 10% val (0.125 of 90%)
x_train, x_val, y_train, y_val = train_test_split(
    x_tv, y_tv, test_size=0.125, stratify=y_tv, random_state=13,
)

print(f"Train: {len(x_train)}, Val: {len(x_val)}, Test: {len(x_test)}")
print(f"Classes: {label_names}")
print(f"\nClass distribution per split:")
for name, y_split in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    counts = np.bincount(y_split, minlength=len(label_names))
    print(f"  {name:5s}: {dict(zip(label_names, counts))}")

## 6. Feature Scaling

RobustScaler uses median and IQR instead of mean/std, making it resistant to outliers. Fitted on training data only — validation and test are transformed, never fitted.

In [ ]:
scaler = RobustScaler()
x_train_s = scaler.fit_transform(x_train)
x_val_s = scaler.transform(x_val)
x_test_s = scaler.transform(x_test)

print(f"Before scaling — train mean: {x_train.mean():.4f}, std: {x_train.std():.4f}")
print(f"After scaling  — train mean: {x_train_s.mean():.4f}, std: {x_train_s.std():.4f}")

## 7. Classifier Comparison (scaled + unscaled)

We train 5 classifiers + 2 DummyClassifier baselines on both scaled and unscaled data. Metrics: accuracy, precision (macro), recall (macro), F1 (macro). Macro averaging weights all classes equally, which penalizes poor performance on minority classes.

In [ ]:
from ml.evaluation import evaluate_classifiers

results_scaled = evaluate_classifiers(x_train_s, x_val_s, y_train, y_val, scaled=True)
results_unscaled = evaluate_classifiers(x_train, x_val, y_train, y_val, scaled=False)

results = pd.concat([results_scaled, results_unscaled]).sort_values("Val F1", ascending=False)
results

### Classifier Comparison Chart

Visual comparison of validation F1 scores across all classifiers and scaling conditions. The dashed line shows the best DummyClassifier baseline — everything below it is worse than guessing.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Separate real classifiers from dummies
real = results[~results["Classifier"].str.startswith("Dummy")].copy()
dummy_best = results[results["Classifier"].str.startswith("Dummy")]["Val F1"].max()

real["Label"] = real["Classifier"] + " (" + real["Scaling"] + ")"
real = real.sort_values("Val F1", ascending=True)

colors = ["#1c83e1" if s == "Scaled" else "#ffa421" for s in real["Scaling"]]
ax.barh(real["Label"], real["Val F1"], color=colors)
ax.axvline(x=dummy_best, color="red", linestyle="--", alpha=0.7, label=f"Dummy baseline ({dummy_best:.3f})")

# Add train F1 as dots for overfitting comparison
ax.scatter(real["Train F1"], real["Label"], color="black", marker="|", s=200, zorder=5, label="Train F1")

ax.set_xlabel("F1 Score (macro)")
ax.set_title("Classifier Comparison: Validation F1 (bars) vs Train F1 (marks)")
ax.legend()
plt.tight_layout()
plt.show()

### Scaling Impact

Does scaling help? Compare Val F1 for each classifier with and without scaling.

In [ ]:
pivot = results[~results["Classifier"].str.startswith("Dummy")].pivot(
    index="Classifier", columns="Scaling", values="Val F1",
)
pivot["Delta (Scaled - Unscaled)"] = pivot["Scaled"] - pivot["Unscaled"]
pivot.sort_values("Scaled", ascending=False)

## 8. Best Model: Test-Set Evaluation

The test set is used **exactly once** — here. We select the best non-dummy classifier by validation F1 (scaled), retrain on the training set, and evaluate on the held-out test set.

In [ ]:
from ml.evaluation import best_model_report, get_classifiers

best_row = results_scaled[
    ~results_scaled["Classifier"].str.startswith("Dummy")
].sort_values("Val F1", ascending=False).iloc[0]

best_name = best_row["Classifier"]
gap = best_row["Train F1"] - best_row["Val F1"]

print(f"Best classifier: {best_name}")
print(f"Val Accuracy:    {best_row['Val Acc']:.1%}")
print(f"Val F1 (macro):  {best_row['Val F1']:.4f}")
print(f"Train-Val Gap:   {gap:.4f} {'⚠️ Overfitting' if gap > 0.15 else '✓ OK'}")

classifiers = get_classifiers()
best_clf = classifiers[best_name]
best_clf.fit(x_train_s, y_train)

report, cm_fig = best_model_report(best_clf, x_test_s, y_test, label_names)
print(f"\nClassification Report ({best_name}):\n")
print(report)

In [ ]:
cm_fig

### Per-Class Performance

Which moods are easiest/hardest to classify? Low F1 for a class means the model struggles — either confusing it with another mood or lacking training examples.

In [ ]:
from sklearn.metrics import classification_report

y_test_pred = best_clf.predict(x_test_s)
report_dict = classification_report(y_test, y_test_pred, target_names=label_names, output_dict=True)

per_class = pd.DataFrame({
    mood: {"Precision": report_dict[mood]["precision"],
           "Recall": report_dict[mood]["recall"],
           "F1": report_dict[mood]["f1-score"],
           "Support": report_dict[mood]["support"]}
    for mood in label_names
}).T.sort_values("F1", ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
x_pos = range(len(per_class))
width = 0.25
ax.bar([p - width for p in x_pos], per_class["Precision"], width, label="Precision", alpha=0.8)
ax.bar(x_pos, per_class["Recall"], width, label="Recall", alpha=0.8)
ax.bar([p + width for p in x_pos], per_class["F1"], width, label="F1", alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(per_class.index, rotation=30)
ax.set_ylabel("Score")
ax.set_title(f"Per-Class Performance ({best_name} on test set)")
ax.legend()
ax.set_ylim(0, 1.1)
# Add support counts as text
for i, (_, row) in enumerate(per_class.iterrows()):
    ax.text(i, max(row["Precision"], row["Recall"], row["F1"]) + 0.03,
            f'n={int(row["Support"])}', ha="center", fontsize=8, color="gray")
plt.tight_layout()
plt.show()

### Error Analysis

Looking at specific misclassifications to understand what the model gets wrong.

In [ ]:
# Get test set keywords
_, kw_test, _, _ = train_test_split(
    keywords, labels, test_size=0.10, stratify=labels, random_state=13,
)

errors = []
for kw, true, pred in zip(kw_test, y_test, y_test_pred):
    if true != pred:
        errors.append({
            "Keyword": kw,
            "True": label_names[true],
            "Predicted": label_names[pred],
        })

error_df = pd.DataFrame(errors)
print(f"Total errors: {len(errors)} / {len(y_test)} ({len(errors)/len(y_test)*100:.1f}%)\n")

# Most common confusion pairs
error_df["Confusion"] = error_df["True"] + " → " + error_df["Predicted"]
print("Most common confusions:")
print(error_df["Confusion"].value_counts().head(10).to_string())
print(f"\nSample misclassified keywords:")
error_df[["Keyword", "True", "Predicted"]].head(15)

## 9. 10-Fold Cross-Validation

Cross-validation on the full dataset gives a more robust estimate of generalization performance than a single train/val split. We use KFold with shuffle.

In [ ]:
from ml.evaluation import run_cross_validation

x_all_s = scaler.fit_transform(embeddings)
cv_scores = run_cross_validation(classifiers[best_name], x_all_s, y)

print(f"10-fold CV ({best_name}):")
print(f"Mean accuracy: {cv_scores.mean():.1%}")
print(f"Std: \u00b1 {cv_scores.std():.1%}")
print(f"Per-fold: {[f'{s:.1%}' for s in cv_scores]}")

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(1, 11), cv_scores, color="#1c83e1", alpha=0.8)
ax.axhline(y=cv_scores.mean(), color="red", linestyle="--", label=f"Mean: {cv_scores.mean():.1%}")
ax.fill_between(range(0, 12), cv_scores.mean() - cv_scores.std(),
                cv_scores.mean() + cv_scores.std(), alpha=0.15, color="red")
ax.set_xlabel("Fold")
ax.set_ylabel("Accuracy")
ax.set_title(f"10-Fold Cross-Validation ({best_name})")
ax.legend()
ax.set_xticks(range(1, 11))
plt.tight_layout()
plt.show()

## 10. KNN Hyperparameter Tuning (k=1..20)

Varying the number of neighbors to find the optimal k. Overfitting manifests as a large gap between train and validation accuracy. The optimal k balances model complexity and generalization.

In [ ]:
from ml.evaluation import knn_hyperparameter_plot

k_fig = knn_hyperparameter_plot(x_train_s, x_val_s, y_train, y_val)
k_fig

## 11. Mood Score Distribution in Production

The trained classifier was applied to 68,462 keywords. These keyword-to-mood predictions are then combined with 3 other signals (genre mapping, overview emotion, review emotion) to produce the final `mood_scores.npy` — a 1.17M × 7 matrix of per-movie mood probabilities.

In [ ]:
mood_scores = np.load(OUTPUT_DIR / "mood_scores.npy")
moods = ["Happy", "Interested", "Surprised", "Sad", "Disgusted", "Afraid", "Angry"]

nonzero = mood_scores[mood_scores.sum(axis=1) > 0]
print(f"Movies with mood scores: {len(nonzero)} / {len(mood_scores)} ({len(nonzero)/len(mood_scores)*100:.1f}%)")

# Dominant mood per movie
dominant = np.argmax(nonzero, axis=1)
dom_counts = pd.Series(dominant).map(lambda i: moods[i]).value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: dominant mood distribution
dom_counts.plot.barh(ax=axes[0], color=[mood_colors[m] for m in dom_counts.index])
axes[0].set_xlabel("Movies")
axes[0].set_title("Dominant Mood per Movie (1.11M movies)")
axes[0].invert_yaxis()

# Right: mood score distributions as violin/box
mood_df = pd.DataFrame(nonzero, columns=moods)
mood_melted = mood_df.melt(var_name="Mood", value_name="Score")
mood_melted = mood_melted[mood_melted["Score"] > 0.01]  # filter near-zero noise
mood_melted.boxplot(column="Score", by="Mood", ax=axes[1], vert=False)
axes[1].set_title("Mood Score Distribution (non-zero, >0.01)")
axes[1].set_xlabel("Score")
plt.suptitle("")

plt.tight_layout()
plt.show()

### Mood Correlation Matrix

How do moods co-occur across 1.11M movies? Negative correlations mean moods are mutually exclusive (e.g., Happy ↔ Afraid). Positive correlations mean moods appear together (e.g., Disgusted + Angry).

In [ ]:
corr = np.corrcoef(nonzero.T)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-0.3, vmax=0.3)
ax.set_xticks(range(7))
ax.set_yticks(range(7))
ax.set_xticklabels(moods, rotation=45, ha="right")
ax.set_yticklabels(moods)
# Annotate cells
for i in range(7):
    for j in range(7):
        ax.text(j, i, f"{corr[i, j]:.2f}", ha="center", va="center", fontsize=9)
ax.set_title("Mood Correlation Matrix (1.11M movies)")
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

## 12. Summary

| Aspect | Result |
|---|---|
| **Best classifier** | MLPClassifier (scaled) |
| **Validation accuracy** | 89.0% |
| **Validation F1 (macro)** | 0.7597 |
| **10-fold CV accuracy** | See above |
| **Overfitting** | Train-Val gap ~0.24 (moderate) |
| **Hardest classes** | Angry, Disgusted, Surprised (low support) |
| **Production coverage** | 68,462 keywords labeled → 94.6% of 1.17M movies have mood scores |

**Key findings:**
- Semantic embeddings (EmbeddingGemma) provide strong signal for mood classification
- MLPClassifier outperforms all others, though overfitting is noticeable
- Class imbalance is the main challenge: rare moods (Angry, Disgusted, Surprised) have lower recall
- Scaling has mixed impact: helps SVC and LogisticRegression, negligible for GaussianNB
- The mood correlation structure in production data is intuitive: Happy↔Afraid are anti-correlated, Disgusted+Angry co-occur